# Phase 06B.03D — Isolated grounding diagnostics

Held-out support labels are opened only after full prediction hashes are locked. Oracle is subset-only and never enters the winner registry.

In [ ]:
from pathlib import Path
import json,math,sys
import numpy as np
import pandas as pd
ROOT=next((p for p in [Path.cwd(),*Path.cwd().parents] if (p/"src").is_dir()),None)
if ROOT is None: raise RuntimeError("Run inside RoadBuddy")
if str(ROOT/"src") not in sys.path: sys.path.insert(0,str(ROOT/"src"))
from roadbuddy_common import save_json
from phase06a_common import sha256_file
OUT=ROOT/"outputs/phase06b/grounding_diagnostics"; OUT.mkdir(parents=True,exist_ok=True)
PRED_LOCK=ROOT/"outputs/phase06b/experiments/full_prediction_hashes.json"; REGISTRY=ROOT/"outputs/phase06b/experiments/novelty_experiment_registry.json"; LABELS=ROOT/"data/phase06b/validation_temporal_support_diagnostic.csv"
if not PRED_LOCK.is_file(): raise RuntimeError("Lock all full prediction hashes before opening diagnostic labels")
lock=json.loads(PRED_LOCK.read_text())
if not LABELS.is_file():
 save_json(OUT/"PHASE06B_03D_STATUS.json",{"status":"awaiting_heldout_support_labels","metrics_available":False,"oracle_registered_as_winner":False}); raise RuntimeError("No held-out labels; no metric will be fabricated")
labels=pd.read_csv(LABELS) # deliberately after the prediction-lock gate
required={"sample_id","support_start_sec","support_end_sec","annotator","adjudication_status"}
if required-set(labels.columns): raise ValueError("Diagnostic support schema/provenance is incomplete")
if (labels.support_start_sec<0).any() or (labels.support_end_sec<labels.support_start_sec).any(): raise ValueError("Invalid diagnostic intervals")
registry=json.loads(REGISTRY.read_text()); arms={a["name"]:a for a in registry["arms"]}


In [ ]:
intervals=labels.groupby(labels.sample_id.astype(str)).apply(lambda x:list(zip(x.support_start_sec.astype(float),x.support_end_sec.astype(float)))).to_dict()
def relevant(times,iv): return np.array([any(start<=t<=end for start,end in iv) for t in times],dtype=bool)
def distance(t,iv): return min(0.0 if start<=t<=end else min(abs(t-start),abs(t-end)) for start,end in iv)
rows=[]; failures=[]
for name,arm in arms.items():
 if name=="L32-F1": continue
 path=Path(arm["predictions_path"]); path=path if path.is_absolute() else ROOT/path
 if not path.is_file() or sha256_file(path)!=arm["predictions_sha256"]: raise ValueError(f"Prediction hash mismatch: {name}")
 pred=pd.read_csv(path); subset=pred[pred.sample_id.astype(str).isin(intervals)]
 recalls={1:[],3:[],8:[]}; aps=[]; ndcgs=[]; distances=[]; briers=[]
 for _,row in subset.iterrows():
  sid=str(row.sample_id); times=np.asarray(json.loads(row.candidate_timestamps_sec),float); scores=np.asarray(json.loads(row.candidate_scores),float); rel=relevant(times,intervals[sid]); order=np.argsort(-scores,kind="stable"); ranked=rel[order]
  for k in recalls: recalls[k].append(float(ranked[:k].any()))
  hits=np.flatnonzero(ranked)+1; aps.append(float(np.mean(np.arange(1,len(hits)+1)/hits)) if len(hits) else 0.0)
  gains=ranked.astype(float)/np.log2(np.arange(len(ranked))+2); ideal=np.sort(rel.astype(float))[::-1]/np.log2(np.arange(len(rel))+2); ndcgs.append(float(gains.sum()/ideal.sum()) if ideal.sum() else 0.0)
  distances.append(distance(times[order[0]],intervals[sid])); prob=np.exp(scores-scores.max()); prob/=prob.sum(); target=rel.astype(float); target=target/target.sum() if target.sum() else target; briers.append(float(np.mean((prob-target)**2)))
  if not ranked[:int(arm["k"])].any(): failures.append({"experiment":name,"sample_id":sid,"failure":"no_relevant_frame_in_top_k"})
 rows.append({"experiment":name,"annotated_rows":len(subset),**{f"recall_at_{k}":float(np.mean(v)) for k,v in recalls.items()},"mAP":float(np.mean(aps)),"nDCG":float(np.mean(ndcgs)),"mean_temporal_distance":float(np.mean(distances)),"score_brier":float(np.mean(briers)),"selector_latency_seconds":float(subset.selector_latency_seconds.mean())})
pd.DataFrame(rows).to_csv(OUT/"grounding_metrics.csv",index=False); pd.DataFrame(failures).to_csv(OUT/"grounding_failure_taxonomy.csv",index=False)
save_json(OUT/"PHASE06B_03D_STATUS.json",{"status":"complete","annotated_subset_only":True,"prediction_lock_sha256":sha256_file(PRED_LOCK),"diagnostic_labels_sha256":sha256_file(LABELS),"oracle_registered_as_winner":False})
rows
